# Percepción de seguridad — Parque La Carolina (2025-2026)
### Usuarios/transeúntes y comerciantes

Reproduce, desde las bases originales, todas las cifras usadas en el producto HTML institucional
(`260831_carolina_v2.html`) y en el borrador Word. Ningún número está escrito a mano: todo se
recalcula desde las 4 bases xlsx.

**Nota metodológica (leer antes de interpretar cualquier comparación 2025→2026):**
el instrumento cambió el enunciado de "en los últimos **6 meses**" (2025) a "en los últimos
**12 meses**" (2026) en casi todas sus preguntas —manteniendo, en ambos años, el mismo texto de
ancla ("de septiembre 2024 a la fecha")—, lo que sugiere un ajuste de redacción no del todo
controlado más que un cambio metodológico deliberado. Afecta percepción general, percepción por
situación específica, tendencia percibida, victimización, armas, incivilidades y presencia
institucional, tanto en usuarios como en comerciantes. Todas las comparaciones 2025→2026 en este
notebook deben leerse como **lecturas direccionales**, no como diferencias estrictamente
controladas. La comparación usuarios-vs-comerciantes **dentro del mismo año (2026)** no tiene este
problema, porque ambos grupos respondieron el mismo instrumento en el mismo levantamiento.

**Bases:**
- `250429_La_Carolina_Pob_Flotante_2025.xlsx` — usuarios/transeúntes 2025 (n=506)
- `260428_La_Carolina_Pob_Flotante_2026_VB.xlsx` — usuarios/transeúntes 2026 (n=505)
- `250429_La_Carolina_Comerciantes_2025.xlsx` — comerciantes 2025 (n=166)
- `260428_La_Carolina_Comerciantes_2026_VB.xlsx` — comerciantes 2026 (n=166)


In [1]:
import pandas as pd, re, json
pd.set_option('display.max_columns', None)

# En Colab: sube los 4 archivos con el panel de archivos, o descomenta para usar el picker:
# from google.colab import files
# uploaded = files.upload()

F25_USR = '250429_La_Carolina_Pob_Flotante_2025.xlsx'
F26_USR = '260428_La_Carolina_Pob_Flotante_2026_VB.xlsx'
F25_COM = '250429_La_Carolina_Comerciantes_2025.xlsx'
F26_COM = '260428_La_Carolina_Comerciantes_2026_VB.xlsx'

df25u = pd.read_excel(F25_USR)
df26u = pd.read_excel(F26_USR, sheet_name='EVPI Carolina Pob Flotante (...')
df25c = pd.read_excel(F25_COM)
df26c = pd.read_excel(F26_COM, sheet_name='EVPI Carolina CAR (2026)')

print('Usuarios  2025:', df25u.shape, '| 2026:', df26u.shape)
print('Comerciantes 2025:', df25c.shape, '| 2026:', df26c.shape)

Usuarios  2025: (506, 222) | 2026: (505, 226)
Comerciantes 2025: (166, 173) | 2026: (166, 186)


## Utilidades comunes

In [2]:
def get_col(df, prefix):
    """Primera columna cuyo nombre empieza con `prefix + ' '` (evita capturar sub-preguntas .1/.2)."""
    for c in df.columns:
        if c.startswith(prefix + ' '):
            return c
    return None

def pct(series, categories=None, dropna=True):
    vc = series.value_counts(normalize=True, dropna=dropna) * 100
    return vc.round(1).to_dict()

def main_prefixed_cols(df, prefix):
    """Columnas 'CC.N ...' o 'SSM.N ...' principales (excluye las sub-columnas de frecuencia .1/.2)."""
    out = []
    pat = re.compile(rf'^{prefix}\.(\d+) ')
    for c in df.columns:
        m = pat.match(c)
        if m:
            out.append((int(m.group(1)), c))
    return out

## 1. Usuarios — percepción general y tendencia

In [3]:
col_psc1_u25 = get_col(df25u, 'PSC.1')
col_psc1_u26 = get_col(df26u, 'PSC.1')
percepcion_u25 = pct(df25u[col_psc1_u25])
percepcion_u26 = pct(df26u[col_psc1_u26])
print('Usuarios 2025 — percepción general:', percepcion_u25)
print('Usuarios 2026 — percepción general:', percepcion_u26)

seguro_u25 = df25u[col_psc1_u25].isin(['1. MUY SEGURO','2. SEGURO']).mean()*100
seguro_u26 = df26u[col_psc1_u26].isin(['1. MUY SEGURO','2. SEGURO']).mean()*100
print(f'\n% seguro/muy seguro usuarios: {seguro_u25:.1f}% (2025) -> {seguro_u26:.1f}% (2026)')

col_tend_u25 = get_col(df25u, 'PSC.9')
col_tend_u26 = get_col(df26u, 'PSC.10')
print('\nTendencia delincuencia 2025:', pct(df25u[col_tend_u25]))
print('Tendencia delincuencia 2026:', pct(df26u[col_tend_u26]))

Usuarios 2025 — percepción general: {'2. SEGURO': 67.4, '3. INSEGURO': 26.1, '1. MUY SEGURO': 4.2, '4. MUY INSEGURO': 2.4}
Usuarios 2026 — percepción general: {'2. SEGURO': 71.3, '3. INSEGURO': 23.8, '1. MUY SEGURO': 4.2, '4. MUY INSEGURO': 0.8}

% seguro/muy seguro usuarios: 71.5% (2025) -> 75.4% (2026)

Tendencia delincuencia 2025: {'2. Esta igual': 61.3, '1. Empeoró': 23.1, '3. Mejoró': 15.6}
Tendencia delincuencia 2026: {'2. Esta igual': 54.9, '3. Mejoró': 25.1, '1. Empeoró': 20.0}


## 2. Usuarios — razones de seguridad/inseguridad (detalle 2026)

In [4]:
seguros_mask = df26u[col_psc1_u26].isin(['2. SEGURO','1. MUY SEGURO'])
inseguros_mask = df26u[col_psc1_u26].isin(['3. INSEGURO','4. MUY INSEGURO'])
n_seg, n_inseg = seguros_mask.sum(), inseguros_mask.sum()
print('n seguros:', n_seg, '| n inseguros:', n_inseg)

seg_cols = [c for c in df26u.columns if c.startswith('PSC.1.1') and '/' in c]
inseg_cols = [c for c in df26u.columns if c.startswith('PSC.1.2') and '/' in c]

print('\n-- Razones de SEGURIDAD (base=seguros) --')
for c in seg_cols:
    v = df26u.loc[seguros_mask, c].sum()
    if v > 0:
        print(f'  {c.split("/")[-1]}: {int(v)} ({v/n_seg*100:.1f}%)')

print('\n-- Razones de INSEGURIDAD (base=inseguros) --')
for c in inseg_cols:
    v = df26u.loc[inseguros_mask, c].sum()
    if v > 0:
        print(f'  {c.split("/")[-1]}: {int(v)} ({v/n_inseg*100:.1f}%)')

n seguros: 381 | n inseguros: 124

-- Razones de SEGURIDAD (base=seguros) --
  1. Afluencia de gente: 243 (63.8%)
  2. No ha pasado nada: 101 (26.5%)
  3. Presencia de la Policía Nacional: 122 (32.0%)
  4. Buena iluminación : 13 (3.4%)
  5. No existe delincuencia : 14 (3.7%)
  6. Presencia de CACMQ: 61 (16.0%)
  7. Existen cámaras de seguridad : 6 (1.6%)
  8. Parque tranquilo: 73 (19.2%)
  9. Presencia de seguridad privada: 49 (12.9%)
  10. ¿Cuál? 1: 29 (7.6%)
  11. ¿Cuál? 2: 1 (0.3%)

-- Razones de INSEGURIDAD (base=inseguros) --
  1. Delincuencia (robos, extorsiones, cualquier tipo de delito): 94 (75.8%)
  2. Venta de drogas: 8 (6.5%)
  3. Consumo de drogas: 16 (12.9%)
  4. Ventas ambulantes: 17 (13.7%)
  5. Consumo de alcohol en el espacio público: 4 (3.2%)
  6. Habitantes de calle (mendicidad): 18 (14.5%)
  7. Falta de alumbrado público: 9 (7.3%)
  8. ¿Cuál? 1: 27 (21.8%)
  9. ¿Cuál? 2: 2 (1.6%)
  10. ¿Cuál? 3: 1 (0.8%)


## 3. Usuarios — percepción por situación específica (PSC.2-8)

In [5]:
situ_prefixes = ['PSC.2','PSC.3','PSC.4','PSC.5','PSC.6','PSC.7','PSC.8']
situaciones = []
for p in situ_prefixes:
    c25, c26 = get_col(df25u, p), get_col(df26u, p)
    v25 = df25u[c25].isin(['3. INSEGURO','4. MUY INSEGURO']).mean()*100
    v26 = df26u[c26].isin(['3. INSEGURO','4. MUY INSEGURO']).mean()*100
    label = c25.split('?')[0].split(' ',1)[1].strip()
    situaciones.append({'label': label, 'y2025': round(v25,1), 'y2026': round(v26,1)})
    print(f'{p}: {label[:55]:<55} {v25:5.1f}% -> {v26:5.1f}%')

PSC.2: Con la presencia de habitantes de calle (mendigos) en e  67.4% ->  61.0%
PSC.3: Con la presencia de vendedores ambulantes en el parque   47.4% ->  40.8%
PSC.4: Transitando por sitios con basura en el parque           64.0% ->  60.4%
PSC.5: En espectáculos públicos como conciertos y/o actividade  37.9% ->  30.7%
PSC.6: En restaurantes/cafeterías/comedores del parque          19.0% ->  13.9%
PSC.7: Caminando sola/o en el día por este parque               34.8% ->  31.7%
PSC.8: Caminando sola/o en la noche por este parque             60.3% ->  61.2%


## 4. Usuarios — incivilidades presenciadas (CC.1-18)

In [6]:
cc25 = dict(main_prefixed_cols(df25u, 'CC'))
cc26 = dict(main_prefixed_cols(df26u, 'CC'))
incivilidades = []
for n in sorted(set(cc25) | set(cc26)):
    c25, c26 = cc25.get(n), cc26.get(n)
    if not (c25 and c26):
        continue
    v25 = (df25u[c25] == '1. Sí').mean()*100
    v26 = (df26u[c26] == '1. Sí').mean()*100
    label = c25.split('?')[0].split(' ',1)[1].strip()
    incivilidades.append({'label': label, 'y2025': round(v25,1), 'y2026': round(v26,1)})
    print(f'{n:>2} {label[:50]:<50} {v25:5.1f}% -> {v26:5.1f}%')

 1 Alguien puso música a un volumen excesivo o hizo m  30.2% ->  19.0%
 2 Grafitis no artisticos que le molestan              24.3% ->  18.2%
 3 Personas que orinan en el parque                    61.7% ->  52.3%
 4 Desechos de animales                                65.4% ->  56.2%
 5 Áreas sin iluminación                               28.9% ->  27.1%
 6 Peleas o riñas                                      18.8% ->  17.6%
 7 Habitantes en situación de calle (mendicidad)       59.1% ->  52.7%
 8 Consumo de alcohol                                  34.2% ->  37.2%
 9 Venta de alcohol                                     9.5% ->   5.1%
10 Consumo de drogas                                   33.2% ->  33.1%
11 Venta de drogas                                     12.1% ->   8.1%
12 Daño a la propiedad pública y privada               25.7% ->  13.7%
13 Fauna Urbana                                        39.5% ->  32.5%
14 Basura y suciedad                                   57.7% ->  49.3%
16 Tra

## 5. Usuarios — victimización, armas y zonas de riesgo percibido\nNota: ventana temporal no comparable (6 vs. 12 meses); leer como fotografía de cada año.

In [7]:
col_vi_u25, col_vi_u26 = get_col(df25u,'VI.1'), get_col(df26u,'VI.1')
col_pa1_u25, col_pa1_u26 = get_col(df25u,'PA.1'), get_col(df26u,'PA.1')
col_pa3_u25, col_pa3_u26 = get_col(df25u,'PA.3'), get_col(df26u,'PA.3')
col_ch4_u25, col_ch4_u26 = get_col(df25u,'CH.4'), get_col(df26u,'CH.4')

victima_u25 = (df25u[col_vi_u25]=='1. Sí').mean()*100
victima_u26 = (df26u[col_vi_u26]=='1. Sí').mean()*100
armas_u25 = (df25u[col_pa1_u25]=='1. Sí').mean()*100
armas_u26 = (df26u[col_pa1_u26]=='1. Sí').mean()*100
cambio_u25 = (df25u[col_ch4_u25]=='1. Sí').mean()*100
cambio_u26 = (df26u[col_ch4_u26]=='1. Sí').mean()*100

print(f'Víctima de delito:      {victima_u25:.1f}% (2025, ventana 6m) -> {victima_u26:.1f}% (2026, ventana 12m)')
print(f'Vio personas con armas: {armas_u25:.1f}% -> {armas_u26:.1f}%')
print(f'Cambió horario visita:  {cambio_u25:.1f}% -> {cambio_u26:.1f}%')

# Tipos de delito
crime_cols_25 = [c for c in df25u.columns if '¿De qué delitos fue víctima?' in c and '/' in c]
crime_cols_26 = [c for c in df26u.columns if '¿De qué delitos fue víctima?' in c and '/' in c]
print('\nDelitos 2025:'); [print(' ', c.split('/')[-1], int(df25u[c].sum())) for c in crime_cols_25 if df25u[c].sum()>0]
print('\nDelitos 2026:'); [print(' ', c.split('/')[-1], int(df26u[c].sum())) for c in crime_cols_26 if df26u[c].sum()>0]

# Zonas
zone_cols_25 = [c for c in df25u.columns if 'mostrar mapa/' in c]
n25 = len(df25u)
zonas_2025 = sorted(({'label': c.split('/')[-1], 'pct': round(df25u[c].sum()/n25*100,1)} for c in zone_cols_25),
                     key=lambda x: -x['pct'])
print('\nZona insegura 2025 (respuesta múltiple):', zonas_2025[:6])

col_zi_26 = 'PSC.11 De acuerdo a este mapa y según su criterio, ¿Cuál es la zona más insegura para Ud., en el parque?. Enc. mostrar mapa'
col_zs_26 = 'PSC.11 De acuerdo a este mapa y según su criterio, ¿Cuál es la zona más segura para Ud., en el parque?. Enc. mostrar mapa'
n26 = len(df26u)
zonas_insegura_2026 = (df26u[col_zi_26].value_counts()/n26*100).round(1)
zonas_segura_2026 = (df26u[col_zs_26].value_counts()/n26*100).round(1)
print('\nZona insegura 2026 (respuesta única):\n', zonas_insegura_2026.head(6))
print('\nZona segura 2026 (pregunta nueva):\n', zonas_segura_2026.head(6))

Víctima de delito:      4.9% (2025, ventana 6m) -> 8.3% (2026, ventana 12m)
Vio personas con armas: 6.9% -> 5.1%
Cambió horario visita:  21.9% -> 23.8%

Delitos 2025:
  VI.2 Robo  (con violencia) 12
  VI.3 Hurto (sin violencia) 5
  VI.4 Robo de vehículo 1
  VI.6 Robo accesorios de vehículo 1
  VI.10 Agresión física 3
  VI.11 Agresión verbal 6
  VI.12 Otros 5

Delitos 2026:
  VI.2 Robo  (con violencia) 26
  VI.3 Hurto (sin violencia) 8
  VI.5 Robo de moto 1
  VI.8 Extorsión 1
  VI.10 Agresión física 2
  VI.11 Agresión verbal 3
  VI.12 Otros 3

Zona insegura 2025 (respuesta múltiple): [{'label': 'L', 'pct': np.float64(23.3)}, {'label': 'F', 'pct': np.float64(20.4)}, {'label': 'B', 'pct': np.float64(19.0)}, {'label': 'E', 'pct': np.float64(18.2)}, {'label': 'A', 'pct': np.float64(15.2)}, {'label': 'C', 'pct': np.float64(13.6)}]

Zona insegura 2026 (respuesta única):
 PSC.11 De acuerdo a este mapa y según su criterio, ¿Cuál es la zona más insegura para Ud., en el parque?. Enc. mostrar mapa

## 6. Usuarios — presencia institucional percibida (SSM)

In [8]:
ssm25 = [c for c in df25u.columns if c.startswith('SSM.') and ' - ' in c]
ssm26 = [c for c in df26u.columns if c.startswith('SSM.') and ' - ' in c]
institucional_u = []
for c25, c26 in zip(ssm25, ssm26):
    v25 = (df25u[c25]=='1. Sí').mean()*100
    v26 = (df26u[c26]=='1. Sí').mean()*100
    label = c25.split(' - ')[0].split(' ',1)[1]
    institucional_u.append({'label': label, 'y2025': round(v25,1), 'y2026': round(v26,1)})
    print(f'{label:<45} {v25:5.1f}% -> {v26:5.1f}%')

Agencia Metropolitana de Control (AMC)         55.1% ->  41.8%
Agencia Metropolitana de Control de Comercio (AMCC)  26.1% ->   9.5%
Patronato San José (UPMSJ)                      7.9% ->   5.7%
Cuerpo de Agentes de Control Metropolitano de Quito (CACMQ)  71.7% ->  66.7%
Cuerpo de Bombero de Quito (CBQ)               15.2% ->  16.0%
Agencia Metropolitana de Tránsito (AMT)        34.8% ->  32.9%
Empresa Metropolitana de Aseo (EMASEO)         78.5% ->  67.3%
Policía Nacional (PN)                          79.6% ->  77.0%
Fuerzas Armadas del Ecuador (FFAA)             20.4% ->  26.5%
Secretaría General de Seguridad Ciudadana y Gestión de Riesgos (SGSCGR)   5.9% ->   4.6%


## 7. Comerciantes — percepción general, en el puesto, y tendencia

In [9]:
col_psc1_c25, col_psc1_c26 = get_col(df25c,'PSC.1'), get_col(df26c,'PSC.1')
col_psc2_c25, col_psc2_c26 = get_col(df25c,'PSC.2'), get_col(df26c,'PSC.2')
col_psc4_c25, col_psc4_c26 = get_col(df25c,'PSC.4'), get_col(df26c,'PSC.4')

print('Percepción general comerciantes 2025:', pct(df25c[col_psc1_c25]))
print('Percepción general comerciantes 2026:', pct(df26c[col_psc1_c26]))
seguro_c25 = df25c[col_psc1_c25].isin(['1. MUY SEGURO','2. SEGURO']).mean()*100
seguro_c26 = df26c[col_psc1_c26].isin(['1. MUY SEGURO','2. SEGURO']).mean()*100
print(f'% seguro/muy seguro comerciantes: {seguro_c25:.1f}% (2025) -> {seguro_c26:.1f}% (2026)')

print('\nPercepción atendiendo el puesto 2025:', pct(df25c[col_psc2_c25]))
print('Percepción atendiendo el puesto 2026:', pct(df26c[col_psc2_c26]))

print('\nTendencia percibida 2025:', pct(df25c[col_psc4_c25]))
print('Tendencia percibida 2026:', pct(df26c[col_psc4_c26]))

Percepción general comerciantes 2025: {'2. SEGURO': 49.4, '3. INSEGURO': 42.8, '4. MUY INSEGURO': 7.2, '1. MUY SEGURO': 0.6}
Percepción general comerciantes 2026: {'2. SEGURO': 48.8, '3. INSEGURO': 43.4, '4. MUY INSEGURO': 6.0, '1. MUY SEGURO': 1.8}
% seguro/muy seguro comerciantes: 50.0% (2025) -> 50.6% (2026)

Percepción atendiendo el puesto 2025: {'2. SEGURO': 63.9, '3. INSEGURO': 26.5, '1. MUY SEGURO': 5.4, '4. MUY INSEGURO': 4.2}
Percepción atendiendo el puesto 2026: {'2. SEGURO': 65.7, '3. INSEGURO': 25.3, '1. MUY SEGURO': 6.0, '4. MUY INSEGURO': 3.0}

Tendencia percibida 2025: {'1. Empeoró': 42.8, '2. Esta igual': 39.8, '3. Mejoró': 17.5}
Tendencia percibida 2026: {'2. Esta igual': 47.6, '1. Empeoró': 27.7, '3. Mejoró': 24.7}


## 8. Comerciantes — razones (2026) e incivilidades

In [10]:
seg_mask_c = df26c[col_psc1_c26].isin(['2. SEGURO','1. MUY SEGURO'])
inseg_mask_c = df26c[col_psc1_c26].isin(['3. INSEGURO','4. MUY INSEGURO'])
n_seg_c, n_inseg_c = seg_mask_c.sum(), inseg_mask_c.sum()
print('n seguros:', n_seg_c, '| n inseguros:', n_inseg_c)

seg_cols_c = [c for c in df26c.columns if c.startswith('PSC.1.1') and '/' in c]
inseg_cols_c = [c for c in df26c.columns if c.startswith('PSC.1.2') and '/' in c]

print('\n-- Razones de SEGURIDAD (comerciantes, base=seguros) --')
for c in seg_cols_c:
    v = df26c.loc[seg_mask_c, c].sum()
    if v > 0:
        print(f'  {c.split("/")[-1]}: {int(v)} ({v/n_seg_c*100:.1f}%)')

print('\n-- Razones de INSEGURIDAD (comerciantes, base=inseguros) --')
for c in inseg_cols_c:
    v = df26c.loc[inseg_mask_c, c].sum()
    if v > 0:
        print(f'  {c.split("/")[-1]}: {int(v)} ({v/n_inseg_c*100:.1f}%)')

print('\n-- Incivilidades presenciadas --')
cc25c = dict(main_prefixed_cols(df25c, 'CC'))
cc26c = dict(main_prefixed_cols(df26c, 'CC'))
incivilidades_c = []
for n in sorted(set(cc25c) | set(cc26c)):
    c25, c26 = cc25c.get(n), cc26c.get(n)
    if not (c25 and c26):
        continue
    v25 = (df25c[c25]=='1. Sí').mean()*100
    v26 = (df26c[c26]=='1. Sí').mean()*100
    label = c25.split('?')[0].split(' ',1)[1].strip()
    incivilidades_c.append({'label': label, 'y2025': round(v25,1), 'y2026': round(v26,1)})
    print(f'{n:>2} {label[:50]:<50} {v25:5.1f}% -> {v26:5.1f}%')

n seguros: 84 | n inseguros: 82

-- Razones de SEGURIDAD (comerciantes, base=seguros) --
  1. Afluencia de gente: 25 (29.8%)
  2. No ha pasado nada: 22 (26.2%)
  3. Presencia de Policía Nacional: 38 (45.2%)
  4. Buena iluminación: 2 (2.4%)
  6. Presencia de CACMQ: 12 (14.3%)
  7. Existen cámaras de seguridad: 1 (1.2%)
  8. Parque tranquilo: 13 (15.5%)
  9. Presencia de seguridad privada: 21 (25.0%)
  10. ¿Cúal?1: 12 (14.3%)
  11. ¿Cúal?2: 1 (1.2%)

-- Razones de INSEGURIDAD (comerciantes, base=inseguros) --
  1. Delincuencia (robos, extorsiones, cualquier tipo de delito): 72 (87.8%)
  2. Venta de drogas: 8 (9.8%)
  3. Consumo de drogas: 19 (23.2%)
  4. Ventas ambulantes: 11 (13.4%)
  5. Consumo de alcohol en el espacio público: 3 (3.7%)
  6. Habitantes de calle (mendicidad): 9 (11.0%)
  7. Falta de alumbrado público: 7 (8.5%)
  8. ¿Cuál?1: 15 (18.3%)

-- Incivilidades presenciadas --
 1 Alguien puso música a un volumen excesivo o hizo m  37.3% ->  34.3%
 2 Grafitis no artísticos que le

## 9. Comerciantes — victimización y presencia institucional

In [11]:
col_vi_c25, col_vi_c26 = get_col(df25c,'VI.1'), get_col(df26c,'VI.1')
victima_c25 = (df25c[col_vi_c25]=='1. Sí').mean()*100
victima_c26 = (df26c[col_vi_c26]=='1. Sí').mean()*100
print(f'Víctima de delito (comerciantes): {victima_c25:.1f}% (2025) -> {victima_c26:.1f}% (2026)')

ssm25c = [c for c in df25c.columns if re.match(r'^SSM\.\d+ ', c)]
ssm26c = [c for c in df26c.columns if re.match(r'^SSM\.\d+ ', c)]
# Emparejar por nombre de institución (el orden/numeración difiere levemente entre años)
def inst_name(c):
    return c.split('(')[0].strip() if '(' in c else c.split(' - ')[0].strip()

d25i = {inst_name(c): (df25c[c]=='1. Sí').mean()*100 for c in ssm25c}
d26i = {inst_name(c): (df26c[c]=='1. Sí').mean()*100 for c in ssm26c}
print('\n-- Presencia institucional (comerciantes) --')
for name in d25i:
    match = next((k for k in d26i if k.split('(')[0].strip() in name or name.split('(')[0].strip() in k), None)
    if match:
        print(f'{name[:40]:<40} {d25i[name]:5.1f}% -> {d26i[match]:5.1f}%')

Víctima de delito (comerciantes): 14.5% (2025) -> 26.5% (2026)

-- Presencia institucional (comerciantes) --
SSM.1 Agencia Metropolitana de Control    59.0% ->  90.4%
SSM.2 Agencia Metropolitana de Control d  31.3% ->  37.3%
SSM.3 Patronato San José                  10.8% ->  21.7%
SSM.4 Cuerpo de Agentes de Control Metro  66.9% ->  85.5%
SSM.5 Cuerpo de Bombero de Quito          30.1% ->  50.0%
SSM.6 Agencia Metropolitana de Tránsito   41.6% ->  29.5%
SSM.7 Empresa Metropolitana de Aseo       93.4% ->  88.6%
SSM.8 Policía Nacional                    86.7% ->  86.1%
SSM.10 Fuerzas Armadas del Ecuador        46.4% ->  75.3%
SSM.11 Secretaría General de Seguridad C   4.8% ->  12.7%
SSM.10 Por favor me puede mencionar tres   0.0% ->   0.0%


## 10. Comparación usuarios vs. comerciantes — solo 2026\nÚnica comparación libre del problema de ventana temporal: mismo instrumento, mismo levantamiento.

In [12]:
print(f'Usuarios seguros/muy seguros 2026:     {seguro_u26:.1f}%  (n={len(df26u)})')
print(f'Comerciantes seguros/muy seguros 2026: {seguro_c26:.1f}%  (n={len(df26c)})')
print(f'Brecha: {seguro_u26 - seguro_c26:.1f} puntos porcentuales')

Usuarios seguros/muy seguros 2026:     75.4%  (n=505)
Comerciantes seguros/muy seguros 2026: 50.6%  (n=166)
Brecha: 24.8 puntos porcentuales


## 11. Ensamblar APP_DATA (mismo objeto que alimenta el HTML)

In [13]:
APP_DATA = {
    'meta': {'n2025': len(df25u), 'n2026': len(df26u), 'parque': 'Parque La Carolina'},
    'percepcionGeneral': {
        'labels': ['Muy seguro','Seguro','Inseguro','Muy inseguro'],
        'y2025': [percepcion_u25.get('1. MUY SEGURO',0), percepcion_u25.get('2. SEGURO',0),
                  percepcion_u25.get('3. INSEGURO',0), percepcion_u25.get('4. MUY INSEGURO',0)],
        'y2026': [percepcion_u26.get('1. MUY SEGURO',0), percepcion_u26.get('2. SEGURO',0),
                  percepcion_u26.get('3. INSEGURO',0), percepcion_u26.get('4. MUY INSEGURO',0)],
    },
    'situaciones': {'items': situaciones},
    'incivilidades': {'items': incivilidades},
    'institucional': {'items': institucional_u},
    'comerciantes': {
        'n2025': len(df25c), 'n2026': len(df26c),
        'incivilidades': {'items': incivilidades_c},
    },
}

with open('app_data_reproducido.json', 'w', encoding='utf-8') as f:
    json.dump(APP_DATA, f, ensure_ascii=False, indent=2)

print('APP_DATA reproducido y guardado en app_data_reproducido.json')
print(json.dumps(APP_DATA['percepcionGeneral'], ensure_ascii=False, indent=2))

APP_DATA reproducido y guardado en app_data_reproducido.json
{
  "labels": [
    "Muy seguro",
    "Seguro",
    "Inseguro",
    "Muy inseguro"
  ],
  "y2025": [
    4.2,
    67.4,
    26.1,
    2.4
  ],
  "y2026": [
    4.2,
    71.3,
    23.8,
    0.8
  ]
}
